In [2]:
from dotenv import load_dotenv
from litellm import completion as litellm_completion
from pathlib import Path
import os
import autogen
from autogen import Agent, AssistantAgent, ConversableAgent, UserProxyAgent
from autogen.agentchat.contrib.capabilities.vision_capability import VisionCapability
from autogen.agentchat.contrib.img_utils import get_pil_image, pil_to_data_uri
from autogen.agentchat.contrib.multimodal_conversable_agent import MultimodalConversableAgent
from autogen.code_utils import content_str

# ── Environment ───────────────────────────────────────────────────────────────
# Get the current Jupyter working directory
PROJECT_ROOT = Path.cwd()

# Load .env from the project root
load_dotenv(PROJECT_ROOT / ".env", override=True)

# Define exports directory
EXPORTS_DIR = PROJECT_ROOT / "exports"

# Check API key
api_key = os.getenv("NVIDIA_API_KEY")

if not api_key:
    raise ValueError("NVIDIA_API_KEY not found. Please check your .env file.")

# NVIDIA NIM OpenAI-compatible base URL
NIM_BASE_URL = "https://integrate.api.nvidia.com/v1"

# Models for AutoGen (OpenAI-compatible)
VISION_MODEL_ID  = "meta/llama-3.2-90b-vision-instruct"
TEXT_MODEL_ID    = "nvidia/nemotron-3.5-lightning-30b-a3b"

In [3]:
from litellm.llms.huggingface.embedding.handler import config
# NVIDIA NIM OpenAI-compatible base URL
NIM_BASE_URL = "https://integrate.api.nvidia.com/v1"

# Models for AutoGen (OpenAI-compatible)
VISION_MODEL_ID  = "meta/llama-3.2-90b-vision-instruct"
TEXT_MODEL_ID    = "nvidia/nemotron-3.5-lightning-30b-a3b"
config_list_v = [
    {
        "model": VISION_MODEL_ID,
        "api_key": os.environ["NVIDIA_API_KEY"],
        "base_url": "https://integrate.api.nvidia.com/v1"
    }
]

config_list_c = [
    {
        "model": TEXT_MODEL_ID ,
        "api_key": os.environ["NVIDIA_API_KEY"],
        "base_url": "https://integrate.api.nvidia.com/v1"
    }
]


In [ ]:
# Image path: adjust this to your actual project structure
image_path = (
    / "src"
    / "exports"
    / "charts"
    / "temp_chart_dea0602a-b255-4c59-84c4-28619e6c5645.png"
)

# Check the file exists
if not image_path.is_file():
    raise FileNotFoundError(f"Image not found: {image_path}")

# Read the local image and convert it to a data URI
pil_image = get_pil_image(str(image_path))
image_uri = pil_to_data_uri(pil_image)


FileNotFoundError: Image not found: c:\Users\roysaw77\Desktop\FYP-Dynamic Dashboarding\FYP-Dynamic-Dashboard-using-LLM\src\exports\src\exports\charts\temp_chart_dea0602a-b255-4c59-84c4-28619e6c5645.png

In [ ]:
image_agent = MultimodalConversableAgent(
    name="image-explainer",
    max_consecutive_auto_reply=10,
    llm_config={"config_list": config_list_v, "temperature": 0.5, "max_tokens": 300},
)

user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="A human admin.",
    human_input_mode="NEVER",  # Try between ALWAYS or NEVER
    max_consecutive_auto_reply=0,
    code_execution_config={
        "use_docker": False
    },  # Please set use_docker=True if docker is available to run the generated code. Using docker is safer than running the generated code directly.
)

user_proxy.initiate_chat(
    image_agent,
    message="""What is the Highest revenue in the plot src/exports/charts/temp_chart_dea0602a-b255-4c59-84c4-28619e6c5645.png""",
)